# 02 — Dynamic biological risk model

## Goal

Translate nine-variable pond trajectories into stress, survival, growth, harvest health, and severe-event labels.

**Used for:** creating the biological target learned by MDN-LSTM and forecast by PHRI.  
**Produces:** `biological_cycles.csv` and `biological_model_parameters.json`.


## Context & Methods

The model aggregates threshold exceedance over each 120-day cycle. DO deficit, ammonia, nitrite, pH deviation, heat, turbidity, alkalinity, and pathogen pressure contribute to stress. Farm management reduces—but does not eliminate—risk.

### Key assumptions

This is a transparent synthetic bridge, not a validated shrimp-growth equation. Refit it with survival, biomass, average body weight, mortality, disease, and harvest records before field use.


In [ ]:
from pathlib import Path
import json
import math
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)
SEED = 20260830
print(f"AQUASURE project root: {ROOT}")


## Data

### 1. Load pond sequences and farm covariates


In [ ]:
pond_path = ARTIFACTS / "pond_timeseries.csv"
if not pond_path.exists(): raise FileNotFoundError("Run notebook 01 first.")
pond = pd.read_csv(pond_path)
farms = pd.read_csv(ARTIFACTS / "farm_profiles.csv")
pond = pond.merge(farms[["farm_id", "management_quality", "stocking_intensity"]], on="farm_id", how="left", validate="many_to_one")
print(f"Loaded {len(pond):,} pond-day observations.")


## Results

### 2. Compute interpretable daily stress components


In [ ]:
sigmoid = lambda x: 1 / (1 + np.exp(-np.clip(x, -30, 30)))
pond["do_stress"] = sigmoid((4.5 - pond["dissolved_oxygen_mg_l"]) / 0.55)
pond["ph_stress"] = sigmoid((np.abs(pond["ph"] - 7.8) - 0.55) / 0.18)
pond["heat_stress"] = sigmoid((pond["temperature_c"] - 31.0) / 0.8)
pond["ammonia_stress"] = sigmoid((pond["ammonia_mg_l"] - 0.32) / 0.09)
pond["nitrite_stress"] = sigmoid((pond["nitrite_mg_l"] - 0.28) / 0.08)
pond["turbidity_stress"] = sigmoid((pond["turbidity_ntu"] - 82) / 13)
pond["alkalinity_stress"] = sigmoid((82 - pond["alkalinity_mg_l"]) / 10)
pond["pathogen_stress"] = pond["pathogen_pressure"]
pond["daily_stress"] = np.clip(
    0.22*pond.do_stress + 0.09*pond.ph_stress + 0.09*pond.heat_stress
    + 0.16*pond.ammonia_stress + 0.12*pond.nitrite_stress + 0.07*pond.turbidity_stress
    + 0.05*pond.alkalinity_stress + 0.20*pond.pathogen_stress, 0, 1,
)
pond[["cycle_id", "day", "daily_stress"]].head()


### 3. Convert cumulative exposure into end-cycle outcomes


In [ ]:
cycle = pond.groupby(["farm_id", "cycle_id"], as_index=False).agg(
    cumulative_stress=("daily_stress", "mean"), peak_stress=("daily_stress", "max"),
    low_do_days=("do_stress", lambda s: int((s > 0.5).sum())),
    high_pathogen_days=("pathogen_stress", lambda s: int((s > 0.65).sum())),
    management_quality=("management_quality", "first"), stocking_intensity=("stocking_intensity", "first"),
)
rng = np.random.default_rng(SEED + 2)
cycle["survival_rate"] = np.clip(0.94 - 0.68*cycle.cumulative_stress - 0.10*cycle.peak_stress + 0.08*cycle.management_quality + rng.normal(0, 0.035, len(cycle)), 0.12, 0.98)
cycle["growth_index"] = np.clip(1.02 - 0.54*cycle.cumulative_stress - 0.08*cycle.stocking_intensity + 0.06*cycle.management_quality + rng.normal(0, 0.025, len(cycle)), 0.25, 1.0)
cycle["harvest_health"] = np.clip(0.62*cycle.survival_rate + 0.38*cycle.growth_index, 0, 1)
H_STAR = float(cycle["harvest_health"].quantile(0.20))
cycle["severe_event"] = (cycle["harvest_health"] < H_STAR).astype(int)
cycle.to_csv(ARTIFACTS / "biological_cycles.csv", index=False)
parameters = {"harvest_threshold_definition": "20th percentile of synthetic baseline harvest-health", "H_star": H_STAR, "label": "harvest_health < H_star"}
(ARTIFACTS / "biological_model_parameters.json").write_text(json.dumps(parameters, indent=2))
print(cycle[["cumulative_stress", "survival_rate", "growth_index", "harvest_health", "severe_event"]].describe().round(4).to_string())
print(f"H*: {H_STAR:.4f}; severe-event rate: {cycle.severe_event.mean():.2%}")


## Checks


In [ ]:
assert cycle[["survival_rate", "growth_index", "harvest_health"]].apply(lambda s: s.between(0, 1).all()).all()
assert 0.15 <= cycle.severe_event.mean() <= 0.25
assert cycle.cycle_id.is_unique
print("PASS — outcomes are bounded, labels are non-degenerate, and cycle grain is unique.")


## Takeaways

This notebook explains what biological harm means, preventing PHRI from becoming an unexplained AI score. Run notebook 03 next.
